<a href="https://colab.research.google.com/github/Ashwini1505-png/codealpha/blob/main/projectmanagement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn sqlalchemy nest-asyncio websockets jinja2


In [2]:
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey, Text
from sqlalchemy.orm import declarative_base, sessionmaker, relationship

# 1. Create the Database Engine
SQLALCHEMY_DATABASE_URL = "sqlite:///./app.db"
engine = create_engine(SQLALCHEMY_DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

# 2. Define our Database Tables (Models)
class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True, index=True)
    username = Column(String, unique=True, index=True)

class Project(Base):
    __tablename__ = "projects"
    id = Column(Integer, primary_key=True, index=True)
    name = Column(String, index=True)
    owner_id = Column(Integer, ForeignKey("users.id"))

class Task(Base):
    __tablename__ = "tasks"
    id = Column(Integer, primary_key=True, index=True)
    title = Column(String)
    status = Column(String, default="To Do") # To Do, In Progress, Done
    project_id = Column(Integer, ForeignKey("projects.id"))
    assigned_to = Column(String, default="Unassigned")

class Comment(Base):
    __tablename__ = "comments"
    id = Column(Integer, primary_key=True, index=True)
    task_id = Column(Integer, ForeignKey("tasks.id"))
    author = Column(String)
    text = Column(Text)

# Create the tables in the database
Base.metadata.create_all(bind=engine)
print("✅ Database and tables created successfully!")

✅ Database and tables created successfully!


In [3]:
from fastapi import FastAPI, Depends, WebSocket, WebSocketDisconnect
from fastapi.responses import HTMLResponse
from typing import List
import json

app = FastAPI()

# Database helper function
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

# --- WEBSOCKET MANAGER (For Real-time Updates) ---
class ConnectionManager:
    def __init__(self):
        self.active_connections: List[WebSocket] = []

    async def connect(self, websocket: WebSocket):
        await websocket.accept()
        self.active_connections.append(websocket)

    def disconnect(self, websocket: WebSocket):
        self.active_connections.remove(websocket)

    async def broadcast(self, message: str):
        for connection in self.active_connections:
            await connection.send_text(message)

manager = ConnectionManager()

@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await manager.connect(websocket)
    try:
        while True:
            # We wait for a message from a user
            data = await websocket.receive_text()
            # And instantly broadcast it to everyone else!
            await manager.broadcast(f"Notification: {data}")
    except WebSocketDisconnect:
        manager.disconnect(websocket)

# --- API ENDPOINTS ---
@app.post("/users/")
def create_user(username: str, db = Depends(get_db)):
    user = db.query(User).filter(User.username == username).first()
    if not user:
        user = User(username=username)
        db.add(user)
        db.commit()
        db.refresh(user)
    return user

@app.post("/projects/")
async def create_project(name: str, owner_id: int, db = Depends(get_db)):
    project = Project(name=name, owner_id=owner_id)
    db.add(project)
    db.commit()
    db.refresh(project)
    await manager.broadcast(f"New project created: {name}")
    return project

@app.post("/tasks/")
async def create_task(title: str, project_id: int, assigned_to: str, db = Depends(get_db)):
    task = Task(title=title, project_id=project_id, assigned_to=assigned_to)
    db.add(task)
    db.commit()
    db.refresh(task)
    await manager.broadcast(f"Task '{title}' assigned to {assigned_to}")
    return task

print("✅ Backend API and WebSockets configured!")

✅ Backend API and WebSockets configured!


In [4]:
html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>Colab Project Manager</title>
    <style>
        body { font-family: Arial, sans-serif; background-color: #f4f5f7; margin: 0; padding: 20px; }
        .container { max-width: 800px; margin: auto; background: white; padding: 20px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }
        h1 { color: #0052cc; }
        .box { border: 1px solid #dfe1e6; padding: 15px; margin-top: 15px; border-radius: 5px; }
        input, button { padding: 10px; margin: 5px 0; width: 100%; box-sizing: border-box; }
        button { background-color: #0052cc; color: white; border: none; cursor: pointer; border-radius: 4px; }
        button:hover { background-color: #0047b3; }
        #notifications { background-color: #e3fcef; padding: 10px; border-radius: 5px; margin-bottom: 15px; font-weight: bold; color: #006644; }
    </style>
</head>
<body>
    <div class="container">
        <h1>📋 Project Management Tool</h1>

        <div id="notifications">Real-time Notifications will appear here...</div>

        <div class="box">
            <h3>1. Auth System</h3>
            <input type="text" id="username" placeholder="Enter your username to login/register">
            <button onclick="login()">Login</button>
            <p id="auth-status" style="color: green;"></p>
        </div>

        <div class="box" id="project-section" style="display:none;">
            <h3>2. Create Project</h3>
            <input type="text" id="projectName" placeholder="New Project Name">
            <button onclick="createProject()">Create Project</button>
        </div>

        <div class="box" id="task-section" style="display:none;">
            <h3>3. Assign Tasks</h3>
            <input type="text" id="taskTitle" placeholder="Task Title (e.g., Design Database)">
            <input type="text" id="assignee" placeholder="Assign to (Username)">
            <button onclick="createTask()">Add Task</button>
        </div>
    </div>

    <script>
        let currentUserId = null;
        let currentProjectId = 1; // Defaulting to 1 for simplicity

        // --- WEBSOCKET CONNECTION ---
        // Dynamically get the correct secure/insecure URL for Colab
        let wsProtocol = window.location.protocol === "https:" ? "wss://" : "ws://";
        let wsUrl = wsProtocol + window.location.host + "/ws";
        let ws = new WebSocket(wsUrl);

        ws.onmessage = function(event) {
            let notifBox = document.getElementById('notifications');
            notifBox.innerText = "🔔 " + event.data;
            // Flash effect
            notifBox.style.backgroundColor = '#fffae6';
            setTimeout(() => { notifBox.style.backgroundColor = '#e3fcef'; }, 1000);
        };

        // --- APP LOGIC ---
        async function login() {
            let username = document.getElementById('username').value;
            let response = await fetch(`/users/?username=${username}`, { method: 'POST' });
            let user = await response.json();
            currentUserId = user.id;

            document.getElementById('auth-status').innerText = `Logged in as ${user.username} (ID: ${user.id})`;
            document.getElementById('project-section').style.display = 'block';
            document.getElementById('task-section').style.display = 'block';

            // Send a real-time message to everyone!
            ws.send(`${user.username} just logged in!`);
        }

        async function createProject() {
            let name = document.getElementById('projectName').value;
            let response = await fetch(`/projects/?name=${name}&owner_id=${currentUserId}`, { method: 'POST' });
            let project = await response.json();
            currentProjectId = project.id;
            alert(`Project '${project.name}' created!`);
        }

        async function createTask() {
            let title = document.getElementById('taskTitle').value;
            let assignee = document.getElementById('assignee').value;
            await fetch(`/tasks/?title=${title}&project_id=${currentProjectId}&assigned_to=${assignee}`, { method: 'POST' });
            alert('Task created! Check notifications.');
        }
    </script>
</body>
</html>
"""

@app.get("/")
def get_home():
    return HTMLResponse(content=html_content)

print("✅ Frontend Interface loaded into the server!")


✅ Frontend Interface loaded into the server!


In [5]:
import nest_asyncio
import uvicorn
import threading
from google.colab import output

# Allow running Uvicorn inside Colab
nest_asyncio.apply()

# Define a function to run the server in the background
def run_app():
    uvicorn.run(app, host="127.0.0.1", port=8000)

# Start the server in a separate thread so it doesn't freeze the notebook
threading.Thread(target=run_app, daemon=True).start()

# Tell Google Colab to give us a clickable link to our server!
print("🚀 Server is running!")
print("👇 CLICK THE LINK BELOW TO OPEN YOUR APP 👇")
output.serve_kernel_port_as_window(8000)


🚀 Server is running!
👇 CLICK THE LINK BELOW TO OPEN YOUR APP 👇
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>